# ⚡ Watching Agents Work: Async and `astream`

## Learning Objectives
In this notebook, you will learn:
1. **Why waiting is the problem** - agents spend almost all their time waiting for replies, not thinking
2. **How to let agents wait together** - turning a 3-second run into a 1-second one, measured
3. **How to watch progress live** - `astream` instead of `ainvoke`, so you see work as it happens
4. **How to watch agents write** - streaming word by word, with two agents writing at once
5. **When async is not worth it** - the cases where it buys you nothing

## Prerequisites
- You have built a small LangGraph system before — nodes, edges, `START`/`END`
- A `.env` at the repository root with credentials for whichever provider `get_llm()` resolves to
- Packages: `langgraph`, `langchain`, `python-dotenv`

> **No new concepts are needed.** If you can read a `for` loop you can read everything here. The only
> two new words are `async` and `await`, and Part 1 explains both by showing what they fix.

## 🤔 The one idea behind this notebook

An agent's time is spent almost entirely **waiting** — for a model to reply, for a search to return,
for a database to answer. It is not busy. It is sitting still.

If three agents each wait one second, you have a choice:

| | What happens | Time |
| --- | --- | --- |
| **One at a time** | Agent A waits. Then B waits. Then C waits. | 3 seconds |
| **All together** | A, B and C wait at the same time. | 1 second |

Nothing got faster. The waiting just **overlapped**. That is the whole of async, and everything
below is a way of arranging it or watching it.

---
## 🔧 Part 0: Setup

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and Model
# ============================================================================

# --- Standard library ---
import asyncio
import operator
import time
from typing import Annotated, TypedDict

# --- Third-party ---
from dotenv import load_dotenv

# --- LangGraph ---
from langgraph.graph import END, START, StateGraph

# --- Project helpers ---
from helpers.utils import get_llm

load_dotenv()

# Only Part 4 uses a real model. Parts 1-3 fake the waiting so the timings are
# exact and cost nothing.
llm = get_llm()

print("✅ Setup complete")

### One thing to know about notebooks

You will write `await something` directly in a cell, with no wrapper around it. That works here
because a notebook is already running the machinery async needs.

The reverse is also true and catches people out: **`asyncio.run(...)` fails in a notebook.** If you
see `RuntimeError: asyncio.run() cannot be called from a running event loop`, delete the
`asyncio.run(` wrapper and just `await` the thing directly.

---
## ⏱️ Part 1: Measuring the waiting

Three researchers, each taking one second. First the slow way, then the fast way, with a stopwatch
on both.

`asyncio.sleep(1)` stands in for a real call to a model or an API. Using a fake keeps the numbers
exact — a real model would vary run to run and hide the point.

In [ ]:
# ============================================================================
# THE WAITING PROBLEM: One at a Time
# ============================================================================


async def research(topic: str) -> str:
    """Pretend to look something up. Takes one second, doing nothing."""
    await asyncio.sleep(1)
    return f"findings about {topic}"


start = time.perf_counter()

# Each `await` here finishes completely before the next line starts.
a = await research("solar")
b = await research("wind")
c = await research("hydro")

print(f"🕐 One at a time: {time.perf_counter() - start:.2f} seconds")
print(f"   Results: {[a, b, c]}")

In [ ]:
# ============================================================================
# THE WAITING PROBLEM: All Together
# ============================================================================
# asyncio.gather starts all three, then waits for all three. Same work, same
# results, same order out - but the waiting overlaps.

start = time.perf_counter()

a, b, c = await asyncio.gather(
    research("solar"),
    research("wind"),
    research("hydro"),
)

print(f"⚡ All together: {time.perf_counter() - start:.2f} seconds")
print(f"   Results: {[a, b, c]}")

### What just happened

Same three calls, same three answers, roughly a third of the time.

Two words did the work:

- **`async def`** marks a function that is *allowed to pause*. Pausing is what lets something else run.
- **`await`** means "pause here until this comes back, and let other work continue meanwhile."

That is all they mean. `asyncio.gather` simply says "start all of these, tell me when they are all done."

---
## 👥 Part 2: The same trick inside a graph

Now the three researchers become three agents in a multi-agent system.

The good news is that you do not have to arrange anything. **If you connect several agents to the
start and make them `async`, LangGraph runs them at the same time for you.** No `gather` required.

The only piece of state here is a shared `notes` list. The `operator.add` on it means each agent's
notes get *added to* the list rather than replacing it — so three agents writing at once do not
overwrite one another.

In [ ]:
# ============================================================================
# PARALLEL AGENTS: Three Specialists, One Graph
# ============================================================================


class ResearchState(TypedDict):
    question: str
    notes: Annotated[list[str], operator.add]  # each agent appends; nobody overwrites


def make_agent(name: str, seconds: float):
    """Build an agent that waits, then files one note."""

    async def agent(state: ResearchState) -> dict:
        await asyncio.sleep(seconds)
        return {"notes": [f"[{name}] looked into '{state['question']}' ({seconds}s)"]}

    return agent


builder = StateGraph(ResearchState)

for agent_name, duration in [("solar", 1.0), ("wind", 1.5), ("hydro", 0.5)]:
    builder.add_node(agent_name, make_agent(agent_name, duration))
    builder.add_edge(START, agent_name)  # all three start together
    builder.add_edge(agent_name, END)

research_team = builder.compile()

print("✅ Three agents wired to start at the same time")

In [ ]:
# ============================================================================
# PARALLEL AGENTS: Run It
# ============================================================================
# `ainvoke` is the async twin of `invoke`. Everything else is identical.

start = time.perf_counter()

result = await research_team.ainvoke({"question": "renewable energy", "notes": []})

elapsed = time.perf_counter() - start
print(f"⚡ Whole team: {elapsed:.2f} seconds")
print(f"   (one at a time would have been {1.0 + 1.5 + 0.5} seconds)")
print()
for note in result["notes"]:
    print(f"   {note}")

### What to notice

The run took about as long as the **slowest** agent (1.5s), not the sum of all three (3.0s). That is
the shape of every parallel system: you wait for the slowest member, and the others are free.

Notice also that the notes came back in a sensible order even though the agents finished at
different times. LangGraph collects the results for you.

---
## 🌊 Part 3: Watching it happen with `astream`

So far every run has been silent until the end. `ainvoke` hands you the finished answer and nothing
before it. For a team that takes thirty seconds, that is thirty seconds of blank screen.

`astream` gives you the same run, but reports as it goes. You ask for what you want to see:

| You want to know | Use | You get |
| --- | --- | --- |
| Who just finished | `stream_mode="updates"` | Only what that agent produced |
| What the whole picture looks like now | `stream_mode="values"` | The full state, after each round |
| What an agent is writing, word by word | `stream_mode="messages"` | Individual pieces of text (Part 4) |

In [ ]:
# ============================================================================
# WATCHING PROGRESS: "updates" - who just finished
# ============================================================================
# Each time an agent completes, we get one small dict: {agent_name: what_it_added}

start = time.perf_counter()

async for update in research_team.astream(
    {"question": "renewable energy", "notes": []},
    stream_mode="updates",
):
    for agent_name, contribution in update.items():
        moment = time.perf_counter() - start
        print(f"   [{moment:4.1f}s] ✅ {agent_name} finished → {contribution['notes'][0]}")

print(f"\n🏁 Done at {time.perf_counter() - start:.1f}s")

Look at the timestamps: **hydro at 0.5s, solar at 1.0s, wind at 1.5s.** They report in the order
they *finish*, not the order they were listed. That is the clearest proof they really did run
together — and it is exactly the progress information you would put on a screen for a waiting user.

In [ ]:
# ============================================================================
# WATCHING PROGRESS: "values" - the whole picture, after each step
# ============================================================================
# Same run, different lens: instead of one agent's contribution, we see the
# entire shared state as it fills up.

start = time.perf_counter()

async for snapshot in research_team.astream(
    {"question": "renewable energy", "notes": []},
    stream_mode="values",
):
    moment = time.perf_counter() - start
    print(f"   [{moment:4.1f}s] notes so far: {len(snapshot['notes'])}")

### Wait — why only two lines?

`updates` gave three reports, one per agent. `values` gave two: empty, then full. That difference
matters and is worth understanding.

LangGraph runs the graph in **rounds**. Three agents wired to the start all belong to the *same*
round, so their results are applied together. `values` reports once per round — so you see the
state before the round and after it, and nothing in between.

`updates` reports per **agent**, which is why it caught all three finishing at different moments.

So: if agents run side by side and you want to see them finish individually, `updates` is the one
that shows it. If they run one after another, both give you a report per agent, because then each
agent is its own round.

### Which one should you use?

- **`updates`** when you want to report progress — "the researcher is done, the writer is starting".
  It is small, it tells you *what changed*, and it reports per agent.
- **`values`** when you want to inspect or debug — you see the whole shared state, once per round.

You can ask for both at once by passing a list: `stream_mode=["updates", "values"]`. Each item then
arrives as a `(which_mode, data)` pair.

---
## ✍️ Part 4: Watching two agents write at the same time

This is the part worth the effort.

`stream_mode="messages"` streams the text an agent is producing as it produces it — word by word,
rather than waiting for the finished paragraph. With two agents running at once, their words arrive
**interleaved**, which makes the concurrency impossible to miss.

This section calls a real model, so it costs a little and takes a few seconds.

In [ ]:
# ============================================================================
# STREAMING TEXT: Two Writers, One Graph
# ============================================================================


class WritingState(TypedDict):
    subject: str
    drafts: Annotated[list[str], operator.add]


async def optimist(state: WritingState) -> dict:
    reply = await llm.ainvoke(f"In two short sentences, say why {state['subject']} is exciting.")
    return {"drafts": [f"[optimist] {reply.content.strip()}"]}


async def skeptic(state: WritingState) -> dict:
    reply = await llm.ainvoke(f"In two short sentences, say why {state['subject']} is overhyped.")
    return {"drafts": [f"[skeptic] {reply.content.strip()}"]}


builder = StateGraph(WritingState)
builder.add_node("optimist", optimist)
builder.add_node("skeptic", skeptic)
builder.add_edge(START, "optimist")
builder.add_edge(START, "skeptic")
builder.add_edge("optimist", END)
builder.add_edge("skeptic", END)

debate = builder.compile()

print("✅ Two writers wired to run at the same time")

In [ ]:
# ============================================================================
# STREAMING TEXT: Watch the Words Arrive
# ============================================================================
# Each item is a pair: the piece of text, and information about where it came
# from. `info["langgraph_node"]` tells us which agent produced it.

print("Streaming (each line shows which agent produced that piece):\n")

pieces_per_agent = {}

async for piece, info in debate.astream(
    {"subject": "self-driving cars", "drafts": []},
    stream_mode="messages",
):
    if not piece.content:
        continue

    who = info["langgraph_node"]
    pieces_per_agent[who] = pieces_per_agent.get(who, 0) + 1

    # Show the first few pieces from each agent so the interleaving is visible.
    if pieces_per_agent[who] <= 8:
        print(f"   {who:>9} | {piece.content!r}")

print()
print("📊 Pieces of text received from each agent:")
for who, count in pieces_per_agent.items():
    print(f"   {who}: {count}")

### What to notice

Scan the left-hand column. The agent names **alternate** — you are watching two agents write at the
same moment, not one after the other.

This is what makes a multi-agent app feel alive instead of frozen. Rather than a spinner for ten
seconds, the user sees both agents working.

In [ ]:
# ============================================================================
# STREAMING TEXT: The Finished Result
# ============================================================================
# Streaming does not replace the final answer - you still get it at the end.

final = await debate.ainvoke({"subject": "self-driving cars", "drafts": []})

for draft in final["drafts"]:
    print(draft)
    print()

---
## 🔀 Part 5: Running whole systems side by side

Everything so far ran agents in parallel *inside* one system. You can also run the **whole system**
several times at once — one run per user, say.

`asyncio.gather` from Part 1 is all you need. Nothing about the graph changes.

In [ ]:
# ============================================================================
# MANY RUNS AT ONCE: Three Questions, One Wait
# ============================================================================

questions = ["solar power", "wind power", "tidal power"]

start = time.perf_counter()

results = await asyncio.gather(
    *[research_team.ainvoke({"question": q, "notes": []}) for q in questions]
)

elapsed = time.perf_counter() - start
print(f"⚡ Three complete team runs: {elapsed:.2f} seconds")
print(f"   (one after another would have been about {1.5 * 3:.1f} seconds)")
print()
for question, outcome in zip(questions, results):
    print(f"   '{question}' → {len(outcome['notes'])} notes")

---
## 🛑 Part 6: When this is not worth doing

Async is not free — it makes code harder to read and harder to debug. Skip it when it buys nothing.

**It buys nothing when the work is a chain.** If the writer needs the researcher's output, the
writer cannot start early. A → B → C takes the same time either way. Async helps things that happen
*side by side*, not things that happen *in order*.

**It buys nothing with a single agent.** One agent waiting is just one agent waiting. Streaming may
still be worth it so the user sees progress, but there is no speed to gain.

**It buys nothing for fast local work.** Async helps with waiting. Arithmetic, string handling and
small data work are not waiting — they are working — and async will make them slightly slower.

The cell below shows the first case: three agents that must run in order, arranged async, gaining
nothing.

In [ ]:
# ============================================================================
# WHEN IT DOES NOT HELP: A Chain of Dependent Agents
# ============================================================================
# Same async code, but each agent needs the previous one's result. The waiting
# cannot overlap because there is nothing to overlap with.


class ChainState(TypedDict):
    text: str


def make_step(name: str):
    async def step(state: ChainState) -> dict:
        await asyncio.sleep(1)
        return {"text": f"{state['text']} → {name}"}

    return step


builder = StateGraph(ChainState)
builder.add_node("research", make_step("researched"))
builder.add_node("write", make_step("written"))
builder.add_node("review", make_step("reviewed"))

builder.add_edge(START, "research")
builder.add_edge("research", "write")   # write must wait for research
builder.add_edge("write", "review")     # review must wait for write
builder.add_edge("review", END)

chain = builder.compile()

start = time.perf_counter()
outcome = await chain.ainvoke({"text": "start"})
print(f"🐌 Chained agents: {time.perf_counter() - start:.2f} seconds — no saving, as expected")
print(f"   {outcome['text']}")

---
## 📝 Summary

### 1. The idea
- Agents mostly **wait**. Async lets them wait at the same time instead of taking turns.
- Nothing runs faster. The waiting overlaps. A team takes as long as its **slowest** member.

### 2. The four things you actually type

| Instead of | Write | For |
| --- | --- | --- |
| `def node(state)` | `async def node(state)` | An agent that waits on something |
| `graph.invoke(...)` | `await graph.ainvoke(...)` | Running it |
| `graph.invoke(...)` | `async for x in graph.astream(...)` | Running it and watching |
| calling one at a time | `await asyncio.gather(a, b, c)` | Several independent things at once |

### 3. Watching a run
- **`stream_mode="updates"`** — who just finished and what they produced. Best for progress messages.
- **`stream_mode="values"`** — the whole shared state, once per round. Best for debugging.
- Agents that start together share one round, so `values` reports them as a single change while `updates` reports each one.
- **`stream_mode="messages"`** — text arriving word by word. Best for making an app feel responsive.
- Agents report in the order they **finish**, which is how you can tell parallelism is real.

### 4. Things that will trip you up
- **`asyncio.run(...)` fails in a notebook.** Just `await` directly instead.
- **Connecting agents to `START` is what makes them parallel** — not the `async` keyword. Agents wired in a chain still run in order.
- **A shared list needs `operator.add`**, or agents writing at the same time will overwrite each other's work.
- **Async does not speed up a chain**, a single agent, or fast local computation.

### Next Steps
- `02_Multi_Agent_Swarm/01_Multi_Agent_Swarm.ipynb` — a real multi-agent system you can apply streaming to.
- `Production_Course_Multi_Agent/07_multi_agent_research_system.ipynb` — dispatches a variable number of workers at runtime and streams the whole thing.
- `03_LangGraph_Fundamentals/02_Core_Capabilities/06_Async_and_Streaming/` — the deeper treatment: timeouts, retries, cancellation, and limiting how many things run at once. Go there once the ideas above are comfortable.